# NexaTel Customer Churn Analytics
## Phase 1 - Data Cleaning & Quality Check

# Objective
The purpose of this phase is to load, inspect, profile, clean and validate the NexaTel datasets before performing exploratory data analysis and churn KPI calculations.

# Dataset
The project contains 24 interconnected CSV tables covering customers, subscriptions, billing, payments, usage, network quality, complaints, support, campaigns and geographical information.

In [60]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

## Define the Raw Data Location

The original NexaTel CSV files are stored in the project's `data/raw` directory.

A dedicated path variable is created so that the datasets can be accessed consistently throughout the notebook. The raw files will be treated as the original source data and will not be modified directly.

Any cleaned versions produced later in the project will be stored separately in the `data/cleaned` directory.

In [61]:
raw_data_path = Path("../data/raw")

raw_data_path

WindowsPath('../data/raw')

In [62]:
csv_files = list(raw_data_path.glob("*.csv"))

len(csv_files)

24

In [63]:
for file in csv_files:
    print(file.name)

billing.csv
cities.csv
complaints.csv
contracts.csv
customers.csv
customer_feedback.csv
data_quality_issue_log.csv
devices.csv
employees.csv
marketing_campaigns.csv
network_quality.csv
payments.csv
plans.csv
plan_history.csv
recharges.csv
regions.csv
retention_campaigns.csv
states.csv
stores.csv
subscriptions.csv
support_tickets.csv
usage_data.csv
usage_sms.csv
usage_voice.csv


## Load the NexaTel Datasets

All 24 CSV files are loaded into Pandas DataFrames and stored in a Python dictionary called `data`.

The filename is used as the dictionary key, allowing each table to be accessed using its table name, for example:

`data["customers"]`

This approach is more efficient and scalable than writing a separate `pd.read_csv()` statement for every table.

In [64]:
data = {}

for file in csv_files:
    table_name = file.stem
    data[table_name] = pd.read_csv(file)

len(data)

24

In [65]:
data.keys()

dict_keys(['billing', 'cities', 'complaints', 'contracts', 'customers', 'customer_feedback', 'data_quality_issue_log', 'devices', 'employees', 'marketing_campaigns', 'network_quality', 'payments', 'plans', 'plan_history', 'recharges', 'regions', 'retention_campaigns', 'states', 'stores', 'subscriptions', 'support_tickets', 'usage_data', 'usage_sms', 'usage_voice'])

THE CODE IN THE CELL BELOW TRUNCATES THE OUTPUT AUTOMATICALLY SO TO PRINT THE COMPLETE OUTPUT, WE NEED TO WRITE AS FOLLOWS:

FOR PRINTING ALL THE COLUMNS, pd.set_option("display.max_columns", None)

FOR PRINTING ALL THE ROWS, pd.set_option("display.max_rows", None)

In [66]:
pd.set_option("display.max_columns", None)
data["customers"].head()

,customer_id,first_name,last_name,gender,date_of_birth,age,email,phone_number,address,city_id,city_name,state_id,pin_code,city_tier,region_id,occupation,annual_income_inr,customer_segment,product_line,plan_id,acquisition_channel,acquisition_date,tenure_months,customer_status,churn_date
0,C0000001,Harish,Shetty,Male,31-03-1988,38,harish.shetty909@yahoo.in,916647481156,"Plot 382, Tilak Lane, Raipur",CT0026,Raipur,ST027,492680,Tier-2,RG04,School Teacher,791058.0,Mass,Prepaid Mobile,PL002,Retail Store,15-01-2026,2,Churned,22-03-2026
1,C0000002,Anjali,Khan,Female,31-03-1974,52,anjali.khan525@rediffmail.com,918116536619,"Door No. 269, Civil Lines Colony, Gurugram",CT0009,Gurugram,ST002,122535,Metro,RG01,Small Business Owner,1385499.0,Premium,Fiber Broadband,PL017,Retail Store,01-01-2014,146,Active,NaN
2,C0000003,Arjun,Hegde,Male,31-03-1978,48,arjun.hegde378@hotmail.com,918958153134,"House No. 320, Garden Nagar, Vadodara",CT0020,Vadodara,ST011,390978,Tier-2,RG02,Auto/Cab Driver,394773.0,Enterprise,Enterprise,PL021,Tele-sales,11-11-2015,1,Churned,22-12-2015
3,C0000004,Vidya,Parmar,Female,31-03-1988,38,vidya.parmar387@hotmail.com,917763920098,"House No. 204, Industrial Street, Nagpur",CT0015,Nagpur,ST010,440881,Tier-2,RG02,Operations Manager,1977532.0,Premium,Prepaid Mobile,PL006,Tele-sales,2014/10/18,137,Active,NaN
4,C0000005,Arjun,Saxena,Male,31-03-1988,38,arjun.saxena410@yahoo.in,916787478273,"Flat 156, Krishna Sector, Warangal",CT0043,Warangal,ST019,506582,Tier-3,RG03,Retired,847300.0,Value,Prepaid Mobile,PL003,Online,20-10-2014,108,Churned,22-10-2023


## Dataset Structure and Initial Profiling

The next step is to perform an initial structural profile of all 24 datasets.

For each table, the following characteristics are examined:

- Number of rows
- Number of columns
- Object/text columns
- Numeric columns
- Datetime columns

This provides an overview of the size and structure of the database and helps identify potential data-type issues that may require attention during the cleaning process.

The profiling results will also help determine which tables and columns require more detailed validation.

In [67]:
inventory = pd.DataFrame([
    {
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    }
    for name, df in data.items()
])

inventory = inventory.sort_values("table").reset_index(drop=True)

inventory

,table,rows,columns
0,billing,115313,9
1,cities,68,8
2,complaints,16584,7
3,contracts,10607,9
4,customer_feedback,28499,7
5,customers,19076,25
6,data_quality_issue_log,20,3
7,devices,19000,8
8,employees,800,9
9,marketing_campaigns,40,9


In [68]:
for name, df in data.items():
    print(f"\n{'=' * 60}")
    print(f"TABLE: {name}")
    print(f"{'=' * 60}")
    print(df.dtypes)


TABLE: billing
invoice_id                  str
customer_id                 str
billing_date                str
billing_period_month        str
base_amount_inr         float64
gst_amount_inr          float64
total_amount_inr        float64
due_date                    str
payment_status              str
dtype: object

TABLE: cities
city_id         str
city_name       str
state_id        str
state_name      str
zone            str
region_id       str
tier            str
pin_prefix    int64
dtype: object

TABLE: complaints
complaint_id       str
customer_id        str
complaint_date     str
complaint_type     str
severity           str
status             str
resolution_date    str
dtype: object

TABLE: contracts
contract_id                 str
customer_id                 str
plan_id                     str
contract_type               str
contract_length_months    int64
start_date                  str
end_date                    str
auto_renew                  str
renewal_status           

In [69]:
dtype_summary = pd.DataFrame([
    {
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "object_columns": df.select_dtypes(include="object").shape[1],
        "numeric_columns": df.select_dtypes(include=np.number).shape[1],
        "datetime_columns": df.select_dtypes(include="datetime").shape[1]
    }
    for name, df in data.items()
])

dtype_summary.sort_values("table").reset_index(drop=True)

,table,rows,columns,object_columns,numeric_columns,datetime_columns
0,billing,115313,9,6,3,0
1,cities,68,8,7,1,0
2,complaints,16584,7,7,0,0
3,contracts,10607,9,8,1,0
4,customer_feedback,28499,7,5,2,0
5,customers,19076,25,20,5,0
6,data_quality_issue_log,20,3,2,1,0
7,devices,19000,8,6,2,0
8,employees,800,9,9,0,0
9,marketing_campaigns,40,9,6,3,0


## Missing Value Analysis

Missing values are an important part of data-quality assessment because incomplete records can affect churn calculations, customer segmentation and downstream analysis.

The missing-value analysis calculates both:

- The number of missing values in each column
- The percentage of records affected

Only columns containing missing values are highlighted so that attention can be focused on fields that require investigation.

Missing values will be investigated first and handled later according to the business meaning of each column rather than being removed automatically.

In [70]:
pd.set_option("display.max_rows", None)
missing_report = pd.concat([
    pd.DataFrame({
        "table": name,
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_percentage": (df.isna().sum().values / len(df) * 100)
    })
    for name, df in data.items()
], ignore_index=True)

missing_report = missing_report.sort_values(
    ["missing_count", "table"],
    ascending=[False, True]
).reset_index(drop=True)

missing_report

,table,column,missing_count,missing_percentage
0,subscriptions,end_date,17017,82.916728
1,customers,churn_date,15560,81.568463
2,support_tickets,resolution_hours,13366,42.445221
3,support_tickets,resolution_date,13366,42.445221
4,complaints,resolution_date,7348,44.307767
5,customers,annual_income_inr,954,5.001048
6,customers,email,745,3.905431
7,customers,address,380,1.992032
8,subscriptions,plan_id,41,0.199776
9,billing,invoice_id,0,0.000000


In [71]:
missing_only = missing_report[
    missing_report["missing_count"] > 0
].reset_index(drop=True)

missing_only

,table,column,missing_count,missing_percentage
0,subscriptions,end_date,17017,82.916728
1,customers,churn_date,15560,81.568463
2,support_tickets,resolution_hours,13366,42.445221
3,support_tickets,resolution_date,13366,42.445221
4,complaints,resolution_date,7348,44.307767
5,customers,annual_income_inr,954,5.001048
6,customers,email,745,3.905431
7,customers,address,380,1.992032
8,subscriptions,plan_id,41,0.199776


## Duplicate Record Analysis

Duplicate records can lead to incorrect customer counts, inflated revenue figures and distorted churn metrics.

The project requirements specifically identify duplicate keys in important tables such as:

- `customers.customer_id`
- `billing.invoice_id`

Duplicate checks are therefore performed at both the **complete-row level** and the **key level**.

At this stage, the duplicates are only being identified and quantified. No records will be removed until their nature and impact have been investigated.

In [72]:
duplicate_report = pd.DataFrame([
    {
        "table": name,
        "duplicate_rows": df.duplicated().sum(),
        "duplicate_percentage": df.duplicated().sum() / len(df) * 100
    }
    for name, df in data.items()
])

duplicate_report = duplicate_report.sort_values(
    "duplicate_rows",
    ascending=False
).reset_index(drop=True)

duplicate_report

,table,duplicate_rows,duplicate_percentage
0,billing,344,0.298318
1,payments,276,0.298824
2,customers,76,0.398406
3,complaints,0,0.000000
4,contracts,0,0.000000
5,customer_feedback,0,0.000000
6,data_quality_issue_log,0,0.000000
7,cities,0,0.000000
8,devices,0,0.000000
9,employees,0,0.000000


In [73]:
customers = data["customers"]

customers["customer_id"].duplicated().sum()

np.int64(76)

THE OUTPUT OF THE CODE BELOW GENERATED WAY TOO MANY NUMBER OF ROWS SO TO JUST PRINT THE FIRST 20 ROWS WE ADDED .head(20)

In [74]:
customers[
    customers["customer_id"].duplicated(keep=False)
].sort_values("customer_id").head(20)


,customer_id,first_name,last_name,gender,date_of_birth,age,email,phone_number,address,city_id,city_name,state_id,pin_code,city_tier,region_id,occupation,annual_income_inr,customer_segment,product_line,plan_id,acquisition_channel,acquisition_date,tenure_months,customer_status,churn_date
97,C0000098,Ayaan,Sheikh,Male,31-03-1977,49,ayaan.sheikh13@rediffmail.com,917063719000,"House No. 229, Old Phase, Siliguri",CT0052,SILIGURI,ST022,734945,Tier-3,RG04,School Teacher,538733.0,Premium,5G,PL013,Retail Store,04-02-2021,61,Active,NaN
19044,C0000098,Ayaan,Sheikh,Male,31-03-1977,49,ayaan.sheikh13@rediffmail.com,917063719000,"House No. 229, Old Phase, Siliguri",CT0052,SILIGURI,ST022,734945,Tier-3,RG04,School Teacher,538733.0,Premium,5G,PL013,Retail Store,04-02-2021,61,Active,NaN
19074,C0000169,Yash,Shinde,Male,01-04-1971,55,yash.shinde18@outlook.com,918905657883,"Flat 66, Lake Nagar, Belagavi",CT0041,Belagavi,ST015,590347,Tier-3,RG03,Sales Executive,506678.0,Value,Prepaid Mobile,PL003,Retail Store,08-11-2020,64,Active,NaN
168,C0000169,Yash,Shinde,Male,01-04-1971,55,yash.shinde18@outlook.com,918905657883,"Flat 66, Lake Nagar, Belagavi",CT0041,Belagavi,ST015,590347,Tier-3,RG03,Sales Executive,506678.0,Value,Prepaid Mobile,PL003,Retail Store,08-11-2020,64,Active,NaN
19025,C0000784,Sita,Hussain,Female,31-03-2008,18,sita.hussain386@rediffmail.com,917836729786,"Apartment 26, Garden Colony, Shimla",CT0061,Shimla,ST007,5845,Tier-3,RG01,Self Employed,626862.0,Mass,Prepaid Mobile,PL002,Retail Store,27-09-2018,90,Active,NaN
783,C0000784,Sita,Hussain,Female,31-03-2008,18,sita.hussain386@rediffmail.com,917836729786,"Apartment 26, Garden Colony, Shimla",CT0061,Shimla,ST007,5845,Tier-3,RG01,Self Employed,626862.0,Mass,Prepaid Mobile,PL002,Retail Store,27-09-2018,90,Active,NaN
1391,C0001392,Rohan,Pradhan,Male,31-03-1980,46,rohan.pradhan706@rediffmail.com,919082312668,"Apartment 19, Lake Phase, Kolkata",CT0005,Kolkata,ST022,700752,Metro,RG04,Auto/Cab Driver,457301.0,Premium,5G,PL013,Online,20-08-2024,19,Active,NaN
19051,C0001392,Rohan,Pradhan,Male,31-03-1980,46,rohan.pradhan706@rediffmail.com,919082312668,"Apartment 19, Lake Phase, Kolkata",CT0005,Kolkata,ST022,700752,Metro,RG04,Auto/Cab Driver,457301.0,Premium,5G,PL013,Online,20-08-2024,19,Active,NaN
1393,C0001394,Preeti,Pawar,Female,31-03-2008,18,preeti.pawar464@yahoo.in,919078014489,"Apartment 300, Tilak Road, Ludhiana",CT0031,Ludhiana,ST003,141305,Tier-2,RG01,Doctor,1263796.0,Value,Postpaid Mobile,PL007,Online,17-02-2022,49,Active,NaN
19020,C0001394,Preeti,Pawar,Female,31-03-2008,18,preeti.pawar464@yahoo.in,919078014489,"Apartment 300, Tilak Road, Ludhiana",CT0031,Ludhiana,ST003,141305,Tier-2,RG01,Doctor,1263796.0,Value,Postpaid Mobile,PL007,Online,17-02-2022,49,Active,NaN


In [75]:
billing = data["billing"]

billing["invoice_id"].duplicated().sum()

np.int64(344)

In [76]:
billing[
    billing["invoice_id"].duplicated(keep=False)
].sort_values("invoice_id").head(20)

,invoice_id,customer_id,billing_date,billing_period_month,base_amount_inr,gst_amount_inr,total_amount_inr,due_date,payment_status
127,INV00000128,C0000030,02-10-2025,10-2025,562.30,101.21,663.51,17-10-2025,Paid
115181,INV00000128,C0000030,02-10-2025,10-2025,562.30,101.21,663.51,17-10-2025,Paid
115278,INV00000652,C0000112,04-07-2025,07-2025,552.95,99.53,652.48,19-07-2025,Overdue
651,INV00000652,C0000112,04-07-2025,07-2025,552.95,99.53,652.48,19-07-2025,Overdue
1219,INV00001220,C0000232,30-01-2026,01-2026,584.68,105.24,689.92,14-02-2026,Overdue
115013,INV00001220,C0000232,30-01-2026,01-2026,584.68,105.24,689.92,14-02-2026,Overdue
1522,INV00001523,C0000272,04-07-2025,07-2025,747.90,134.62,882.52,19-07-2025,Paid
115081,INV00001523,C0000272,04-07-2025,07-2025,747.90,134.62,882.52,19-07-2025,Paid
1599,INV00001600,C0000282,30-01-2026,01-2026,486.59,87.59,574.18,14-02-2026,Overdue
115275,INV00001600,C0000282,30-01-2026,01-2026,486.59,87.59,574.18,14-02-2026,Overdue


### Primary Key / Business Key Validation

The tables identified during the initial duplicate analysis are examined more closely to determine whether their duplicate records also contain repeated key values.

The following keys are checked:

- `customers` → `customer_id`
- `billing` → `invoice_id`
- `payments` → `payment_id`

Comparing complete duplicate rows with duplicate key counts helps determine whether the issue is caused by repeated records or by multiple different records sharing the same key.

The results will be used later when deciding how these records should be cleaned.

In [77]:
key_duplicate_check = pd.DataFrame({
    "table": ["customers", "billing", "payments"],
    "key": ["customer_id", "invoice_id", "payment_id"],
    "duplicate_key_count": [
        data["customers"]["customer_id"].duplicated().sum(),
        data["billing"]["invoice_id"].duplicated().sum(),
        data["payments"]["payment_id"].duplicated().sum()]
})

key_duplicate_check

,table,key,duplicate_key_count
0,customers,customer_id,76
1,billing,invoice_id,344
2,payments,payment_id,276


### Confirm the Nature of the Duplicate Records

A final comparison is performed between complete duplicate rows and duplicate values in the corresponding keys.

If both counts are equal, it indicates that the repeated keys are associated with complete duplicate records rather than different records sharing the same key.

For the affected tables, this comparison will help establish whether the duplicates can later be safely handled by retaining one occurrence of each repeated record.

In [78]:
for table in ["customers", "billing", "payments"]:
    df = data[table]
    
    print(f"{table}:")
    print("  Complete duplicate rows:", df.duplicated().sum())
    print("  Duplicate rows based on key:", df[df.columns[0]].duplicated().sum())
    print()

customers:
  Complete duplicate rows: 76
  Duplicate rows based on key: 76

billing:
  Complete duplicate rows: 344
  Duplicate rows based on key: 344

payments:
  Complete duplicate rows: 276
  Duplicate rows based on key: 276



##     Review the Data Quality Issue Log

The project includes a dedicated `data_quality_issue_log` table containing the intentionally injected data-quality issues and the number of affected records.

This log is used as a reference checklist during validation.

Rather than assuming that every potential data problem exists, the analysis will systematically investigate the issues documented in the log and compare the detected records with the expected `rows_affected` values.

The issue log contains **20 data-quality issues** across the `customers`, `billing`, `payments`, `subscriptions` and `usage_data` tables.

No changes will be made to the issue log because it serves as a validation reference throughout Phase 1.

In [79]:
quality_log = data["data_quality_issue_log"]

quality_log

,table,issue_type,rows_affected
0,customers,missing_email,760
1,customers,missing_address,380
2,customers,missing_income,950
3,customers,invalid_phone,228
4,customers,invalid_email,285
5,customers,incorrect_pin,190
6,customers,misspelled_city,380
7,customers,inconsistent_capitalisation,570
8,customers,impossible_age,38
9,customers,mixed_date_format,380


### Organize the Data Quality Checklist

The quality log is sorted by table and issue type to create a structured checklist for the cleaning process.

This makes it easier to work through the issues systematically, beginning with the customer data and then moving to billing, payments, subscriptions and usage data.

The `rows_affected` column provides an expected benchmark that can be compared with the results of our independent validation checks.

In [80]:
quality_log.sort_values(
    ["table", "issue_type"]
).reset_index(drop=True)

,table,issue_type,rows_affected
0,billing,duplicate_invoices,344
1,billing,future_dates,229
2,billing,incorrect_gst,1149
3,billing,negative_invoice_amount,229
4,customers,duplicate_customers,76
5,customers,impossible_age,38
6,customers,inconsistent_capitalisation,570
7,customers,incorrect_pin,190
8,customers,invalid_email,285
9,customers,invalid_phone,228


### Validate Customer Age

The `age` column is examined for impossible or inconsistent values.

The initial validation checks the statistical range of the column and identifies values that fall outside a reasonable age range.

The data-quality issue log reports **38 records affected by the `impossible_age` issue**, so the validation results will be compared against this benchmark.

The objective is to identify the affected records without modifying the original data at this stage.

In [81]:
customers["age"].describe()

count    19076.000000
mean        36.069721
std         12.125744
min         -5.000000
25%         28.000000
50%         36.000000
75%         44.000000
max        220.000000
Name: age, dtype: float64

### Data Quality Investigation: Validating Customer Age

During the validation of the `age` column, the initial check identified only 23 records with obviously impossible values such as `-5`, `130`, and `220`. However, the `data_quality_issue_log` indicated that **38 records** were intentionally affected by the `impossible_age` issue.

To investigate this discrepancy, the `date_of_birth` column was converted from text into a proper datetime format using `pd.to_datetime()`. The customer's age was then independently recalculated from the year of birth.

An initial attempt used an exact age calculation based on whether the customer's birthday had already occurred by the reference date (31-03-2026). This produced 4,491 apparent age mismatches. Further investigation showed that these were **not data-quality errors**. The dataset's existing `age` values follow a simpler calculation:

> **Age = 2026 − Year of Birth**

For example, a customer born on `01-04-1991` has a recorded age of `35`, even though their birthday falls one day after the reference date. Therefore, the dataset does not use an exact birthday-based age calculation.

After aligning the validation logic with the calculation convention used in the dataset, the comparison identified exactly **38 age mismatches**, matching the `impossible_age` count recorded in the data-quality issue log.

**Conclusion:** The 38 mismatched records are confirmed as the intentionally injected `impossible_age` issues. This investigation highlights the importance of understanding the dataset's underlying calculation logic before classifying records as erroneous.

In [82]:
customers[
    (customers["age"] < 0) | (customers["age"] > 100)
][["customer_id", "date_of_birth", "age"]]

,customer_id,date_of_birth,age
751,C0000752,31-03-1989,220
834,C0000835,31-03-1988,220
2917,C0002918,31-03-1984,130
3026,C0003027,01-04-1971,220
3822,C0003823,01-04-1983,220
3824,C0003825,01-04-1979,220
4639,C0004640,31-03-1982,130
5159,C0005160,31-03-1969,130
5290,C0005291,31-03-2006,130
5833,C0005834,31-03-1982,-5


In [83]:
invalid_age = customers[
    (customers["age"] < 0) | (customers["age"] > 100)
]

invalid_age.shape[0]

23

In [84]:
invalid_age["age"].value_counts().sort_index()

age
-5      6
 130    8
 220    9
Name: count, dtype: int64

In [89]:
customers["date_of_birth_parsed"] = pd.to_datetime(
    customers["date_of_birth"],
    dayfirst=True,
    errors="coerce"
)

In [90]:
customers[["date_of_birth", "date_of_birth_parsed"]].head()

,date_of_birth,date_of_birth_parsed
0,31-03-1988,1988-03-31
1,31-03-1974,1974-03-31
2,31-03-1978,1978-03-31
3,31-03-1988,1988-03-31
4,31-03-1988,1988-03-31


In [93]:
reference_date = pd.Timestamp("2026-03-31")

customers["calculated_age"] = (
    reference_date.year - customers["date_of_birth_parsed"].dt.year
)

In [96]:
reference_date = pd.Timestamp("2026-03-31")

customers["calculated_age"] = (
    reference_date.year
    - customers["date_of_birth_parsed"].dt.year
    - (
        (reference_date.month < customers["date_of_birth_parsed"].dt.month)
        |
        (
            (reference_date.month == customers["date_of_birth_parsed"].dt.month)
            &
            (reference_date.day < customers["date_of_birth_parsed"].dt.day)
        )
    ).astype(int)
)    

In [99]:
age_mismatch = customers[
    customers["age"] != customers["calculated_age"]
][
    ["customer_id", "date_of_birth", "age", "calculated_age"]
]

age_mismatch.shape[0]

4491

In [100]:
customers[["date_of_birth", "age"]].head(20)

,date_of_birth,age
0,31-03-1988,38
1,31-03-1974,52
2,31-03-1978,48
3,31-03-1988,38
4,31-03-1988,38
5,31-03-1990,36
6,31-03-2000,26
7,01-04-1991,35
8,31-03-1996,30
9,31-03-2006,20


In [104]:
customers["date_of_birth_parsed"].min(), customers["date_of_birth_parsed"].max()

(Timestamp('1947-04-01 00:00:00'), Timestamp('2008-03-31 00:00:00'))

In [107]:
customers[customers["age"].isin([-5, 130, 220])][
    ["customer_id", "date_of_birth", "age"]
].head(20)

,customer_id,date_of_birth,age
751,C0000752,31-03-1989,220
834,C0000835,31-03-1988,220
2917,C0002918,31-03-1984,130
3026,C0003027,01-04-1971,220
3822,C0003823,01-04-1983,220
3824,C0003825,01-04-1979,220
4639,C0004640,31-03-1982,130
5159,C0005160,31-03-1969,130
5290,C0005291,31-03-2006,130
5833,C0005834,31-03-1982,-5


In [108]:
age_difference = customers["age"] - customers["calculated_age"]

age_difference.value_counts().sort_index()

-50         1
-49         1
-48         1
-46         1
-43         1
-40         2
-39         1
-35         3
-33         1
-32         2
-29         2
-27         1
-24         1
-23         1
-18         1
-17         1
 0      14585
 1       4453
 73         1
 74         1
 86         1
 88         2
 108        1
 110        1
 112        1
 166        1
 168        1
 170        1
 174        1
 178        1
 182        1
 183        1
 195        2
Name: count, dtype: int64

In [109]:
customers[
    customers["age"] != customers["calculated_age"]
][
    ["customer_id", "date_of_birth", "age", "calculated_age"]
].head(30)

,customer_id,date_of_birth,age,calculated_age
7,C0000008,01-04-1991,35,34
10,C0000011,01-04-1979,47,46
13,C0000014,01-04-1983,43,42
14,C0000015,01-04-1991,35,34
15,C0000016,01-04-1983,43,42
16,C0000017,01-04-2003,23,22
18,C0000019,01-04-1987,39,38
22,C0000023,01-04-1999,27,26
24,C0000025,01-04-2007,19,18
27,C0000028,01-04-2003,23,22


In [110]:
customers["date_of_birth"].value_counts().head(20)

date_of_birth
31-03-2008    1465
31-03-1989     653
31-03-1990     648
31-03-1988     645
01-04-1987     638
31-03-1992     629
31-03-1993     627
31-03-1994     606
31-03-1985     600
01-04-1995     583
31-03-1986     579
01-04-1991     575
01-04-1983     567
31-03-1997     542
31-03-1996     530
31-03-1984     525
31-03-1998     512
31-03-1981     495
31-03-1982     488
31-03-2000     475
Name: count, dtype: int64

In [111]:
customers["calculated_age"] = (
    reference_date.year
    - customers["date_of_birth_parsed"].dt.year
)

In [112]:
age_mismatch = customers[
    customers["age"] != customers["calculated_age"]
][
    ["customer_id", "date_of_birth", "age", "calculated_age"]
]

age_mismatch.shape[0]

38

### Inspect the Affected Age Records

The validation step has identified 38 customer records where the recorded age does not match the age calculated from the customer's year of birth.

Before correcting these records, the affected rows are inspected to understand the nature of the errors and determine whether the correct age can be reliably reconstructed from the available `date_of_birth` information.

No values will be modified during this inspection.

The affected records show clearly invalid recorded ages such as `-5`, `0`, `1`, `130`, and `220`.

And since the correct age can be reconstructed from the customer's date of birth using the dataset's established age calculation logic, the calculated ages will be checked before applying the correction.

No original values will be overwritten until this validation is complete.

In [113]:
age_mismatch.sort_values("age")[[
    "customer_id",
    "date_of_birth",
    "age",
    "calculated_age"
]]

,customer_id,date_of_birth,age,calculated_age
7018,C0007019,31-03-1996,-5,30
5833,C0005834,31-03-1982,-5,44
15136,C0015137,31-03-2004,-5,22
8666,C0008667,31-03-2002,-5,24
13762,C0013763,01-04-1987,-5,39
13922,C0013923,31-03-1981,-5,45
2175,C0002176,01-04-1979,0,47
1604,C0001605,31-03-2002,0,24
15287,C0015288,31-03-2008,0,18
9326,C0009327,01-04-2007,1,19


In [114]:
age_mismatch[[
    "customer_id",
    "date_of_birth",
    "age",
    "calculated_age"
]].sort_values("customer_id")

,customer_id,date_of_birth,age,calculated_age
506,C0000507,31-03-1992,1,34
751,C0000752,31-03-1989,220,37
834,C0000835,31-03-1988,220,38
868,C0000869,31-03-1990,1,36
1604,C0001605,31-03-2002,0,24
2175,C0002176,01-04-1979,0,47
2917,C0002918,31-03-1984,130,42
3026,C0003027,01-04-1971,220,55
3822,C0003823,01-04-1983,220,43
3824,C0003825,01-04-1979,220,47


In [115]:
customers.loc[
    age_mismatch.index,
    "age"
] = customers.loc[
    age_mismatch.index,
    "calculated_age"
]

In [116]:
age_mismatch_after = customers[
    customers["age"] != customers["calculated_age"]
][
    ["customer_id", "date_of_birth", "age", "calculated_age"]
]

age_mismatch_after.shape[0]

0